In [1]:
from typing import TypedDict, Annotated, List, Union
from langgraph.graph.message import add_messages, StateGraph
from langchain.messages import AIMessage
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import datetime
from langgraph.graph import END
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


In [2]:
class AgentState(TypedDict):
    input:str
    intermediate_step:Annotated[List, add_messages]
    output:Union[AIMessage,None]

In [3]:
prompt_template=ChatPromptTemplate.from_messages([
    ("system","You are an AI assistant, who uses as less tokens as possible to help the user"
     "Only use tools if you have to"),
    ("human","{input}")
])

In [4]:
@tool
def get_system_time(format:str="%Y-%m-%d %H:%M:%S"):
    """
    Returns the current date and time.
    
    CRITICAL INSTRUCTION:
    To call this tool, you MUST output a perfectly formatted JSON object with exactly one key called "format_string".
    Sample output:
    {"format_string": "%Y-%m-%d %H:%M:%S"}
    """
    dtime=datetime.datetime.now()
    fdtime=dtime.strftime(format)
    return fdtime

tools=[get_system_time]

# llm = ChatGroq(model="llama-3.1-8b-instant")
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
# llm = ChatGroq(model="groq/compound")
# llm = ChatGroq(model="llama-3.3-70b-versatile")


llm_with_tools=llm.bind_tools(tools=tools)

In [ ]:
responder_chain= prompt_template | llm_with_tools

def responder_node(state:AgentState):
    result=responder_chain.invoke(state)
    return {"output":result}

def should_continue(state:AgentState):
    lastAIMessage=state["output"]
    if isinstance(lastAIMessage,AIMessage) and hasattr(lastAIMessage,"tool_calls") and len(lastAIMessage.tool_calls)>0:
        return "tool_node"
    else:
        return END

def tool_node(state:AgentState):
    lastAIMessage=state["output"]
    for i,tool_list in enumerate(lastAIMessage.tool_calls):
        tool_args=tool_list[i]["args"]
        tool_name=tool_list[i]["name"]
        tool_to_run=next((t for t in tools if tool_name==t.name),None)

    if tool_to_run:
        tool_result=tool_to_run.invoke(tool_args)
    else:
        tool_result=f"Use only {tools} as tools"

    return {"intermediatestep":[tool_name,tool_result]}

In [6]:
graph=StateGraph(AgentState)
graph.add_node("responder_node", responder_node)
graph.add_node("tool_node", tool_node)
graph.set_entry_point("responder_node")
graph.add_edge("tool_node","responder_node")
graph.add_conditional_edges("responder_node",should_continue)
app=graph.compile()

In [8]:
answer=app.invoke({
    "input":"what's the current time",
    "intermediate_step":[],
    "output":None
})

ChatGoogleGenerativeAIError: Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

In [ ]:
print(answer["output"].tool_calls[0])

{'name': 'get_system_time', 'args': {}, 'id': 'qgpksxxx0', 'type': 'tool_call'}
